# Stage 5a — Sentiment Classification (Tier 1)

Scores every chunk with `cardiffnlp/twitter-roberta-base-sentiment-latest`.
Outputs soft probabilities `sent_neg`, `sent_neu`, `sent_pos` per chunk.

**Input datasets needed:**
- `ns-sentiment-chunks-v3` — `submissions_chunks.parquet`, `comments_chunks.parquet`

**Output:** `chunk_sentiment.parquet` — columns: `chunk_id`, `sent_neg`, `sent_neu`, `sent_pos`

**Setup:** GPU T4 x2 · Save & Run All

In [ ]:
# Cell 1 — Discover input paths
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
# Cell 2 — Config  <- UPDATE PATHS AFTER RUNNING CELL 1
SUBMISSIONS_CHUNKS = '/kaggle/input/<your-dataset>/submissions_chunks.parquet'
COMMENTS_CHUNKS    = '/kaggle/input/<your-dataset>/comments_chunks.parquet'
OUT_DIR            = '/kaggle/working'

MODEL_NAME   = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
BATCH_SIZE   = 256    # safe for T4 at 512-token truncation
CHECKPOINT_N = 50_000 # save partial parquet every N chunks

In [ ]:
# Cell 3 — Install
!pip install transformers -q

import torch
import numpy as np
import pandas as pd
import time
from pathlib import Path
from transformers import pipeline

print('Imports OK')

In [ ]:
# Cell 4 — Load model + tokenizer
device = 0 if torch.cuda.is_available() else -1
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

classifier = pipeline(
    'sentiment-analysis',
    model=MODEL_NAME,
    top_k=None,       # replaces deprecated return_all_scores=True
    device=device,
    truncation=True,
    max_length=512,
)
print('Model loaded')

# Verify label set — top_k=None returns list-of-lists: [[{label, score}, ...]]
test_out = classifier(['test'])
labels = sorted(s['label'] for s in test_out[0])
print(f'Label set: {labels}')  # expect: ['negative', 'neutral', 'positive']

In [ ]:
# Cell 5 — Load and combine chunks
print('Loading chunk parquets ...')
sub = pd.read_parquet(SUBMISSIONS_CHUNKS, columns=['chunk_id', 'text'])
com = pd.read_parquet(COMMENTS_CHUNKS,    columns=['chunk_id', 'text'])
df  = pd.concat([sub, com], ignore_index=True)
del sub, com
print(f'Total chunks: {len(df):,}')

# Drop nulls / empty strings
df['text'] = df['text'].fillna('').str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)
print(f'After null filter: {len(df):,}')

In [ ]:
# Cell 6 — Batch inference with checkpointing
texts     = df['text'].tolist()
chunk_ids = df['chunk_id'].tolist()
n         = len(texts)

records          = []
checkpoint_files = []
t0               = time.time()

for start in range(0, n, BATCH_SIZE):
    batch_texts = texts[start : start + BATCH_SIZE]
    batch_ids   = chunk_ids[start : start + BATCH_SIZE]

    raw = classifier(batch_texts, batch_size=BATCH_SIZE)

    for cid, scores in zip(batch_ids, raw):
        score_map = {s['label']: s['score'] for s in scores}
        records.append({
            'chunk_id': cid,
            'sent_neg': score_map.get('negative', np.nan),
            'sent_neu': score_map.get('neutral',  np.nan),
            'sent_pos': score_map.get('positive', np.nan),
        })

    done = start + len(batch_texts)

    # Checkpoint every CHECKPOINT_N rows (and on the final batch)
    if done % CHECKPOINT_N < BATCH_SIZE or done >= n:
        cp_path = f'{OUT_DIR}/sentiment_checkpoint_{done}.parquet'
        pd.DataFrame(records).to_parquet(cp_path, index=False)
        checkpoint_files.append(cp_path)
        elapsed = time.time() - t0
        rate    = done / elapsed if elapsed > 0 else 0
        eta     = (n - done) / rate if rate > 0 else 0
        print(f'{done:>7,} / {n:,}  |  {rate:.0f} chunks/s  |  ETA {eta/60:.1f} min')

print(f'\nDone — {len(records):,} chunks scored in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 7 — Merge checkpoints and save
cp_dfs   = [pd.read_parquet(f) for f in checkpoint_files]
sent_df  = pd.concat(cp_dfs, ignore_index=True).drop_duplicates(subset='chunk_id')

print(f'Final rows:    {len(sent_df):,}')
print(f'Null sent_neg: {sent_df["sent_neg"].isna().sum():,}')

out_path = f'{OUT_DIR}/chunk_sentiment.parquet'
sent_df.to_parquet(out_path, index=False)
print(f'Saved {out_path}')

In [ ]:
# Cell 8 — Quality summary
print('── Score distributions ──')
for col in ['sent_neg', 'sent_neu', 'sent_pos']:
    print(f'\n{col}:')
    print(sent_df[col].describe().round(4))

# Majority-label distribution
label_map = {'sent_neg': 'negative', 'sent_neu': 'neutral', 'sent_pos': 'positive'}
sent_df['majority_label'] = sent_df[['sent_neg', 'sent_neu', 'sent_pos']].idxmax(axis=1).map(label_map)
print('\n── Majority label counts ──')
vc = sent_df['majority_label'].value_counts()
for lbl, cnt in vc.items():
    print(f'{lbl:<10} {cnt:>8,}  ({cnt/len(sent_df)*100:.1f}%)')

# Confidence: how decisive is the model?
sent_df['max_score'] = sent_df[['sent_neg', 'sent_neu', 'sent_pos']].max(axis=1)
print('\n── Confidence (max score) ──')
print(sent_df['max_score'].describe().round(4))
low_conf = (sent_df['max_score'] < 0.5).sum()
print(f'Low-confidence (<0.5): {low_conf:,} ({low_conf/len(sent_df)*100:.1f}%)')
print('\nDone. Download chunk_sentiment.parquet from /kaggle/working.')